In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
print(sys.platform)
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


linux


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [ ]:
from pprint import pprint

pprint(response)

## Online MCP

In [66]:

client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [67]:
agent = create_agent(
    model=model,
    tools=tools
)

In [68]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='240de0c0-9189-45d6-8398-15ba8b4e4eb0'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, '__gemini_function_call_thought_signatures__': {'call_759899': 'Es4CCssCAWkUfRPodApurnPffagvkfyCVa4tDKtSxmSxTmoEoIt6vgi9FSs05yixSLfNtBkzeKCzBY+2bWZtkoCEUabUmIGGNs+edpUjsq79rhJBpCIHMC3pHtHFI8vf3IKf/9pzVT2A1skZxvsCxuEeQF3VNoTgBGIhXj4gC2wYeyT48YyIUtb/csxko6gNUjryIoqE3sN+FGiSL9Rsjlk37rQR+wqeYvlJ3HOF5E65mxMMkm3lDw4RzuSuPK8/97QIBb3yZRO/wZuLl58vzclAy9wYhXulrysNXkSY54W4MtSM7OJRwKYA3xfH1o4EJ364PHbCIYLH6PuBvrE5ttjPgAYLCuvi7v54ZzWkf01P5rU3Sw/PALTv3dlAzwUOCjGCKFnGQlaIqYV3PEJxsJByiqDj65095DtAPhjt0CfbJ0YbI5oWDhCCWnc4Pu36hg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a16e2e-bccf-7ea1-987c-350e6dbb

## Trip MCP

In [6]:

client = MultiServerMCPClient(
    {
        "instavision": {
          "transport": "stdio",
          "command": "npx",
          "args": [
            "-y",
            "instavision-mcp"
          ],
          "env": {
            "INSTAVISION_API_KEY": "iv_sk_wlK6euTZufDKhFZKtBqU-Y54YqYfq3Kj8R6fmXK_qzE"
          }
        }
    }
)

tools = await client.get_tools()

In [7]:
agent = create_agent(
    model=model,
    tools=tools
)

## Unable to retain conversation history in the following two calls

In [11]:
from langchain.messages import HumanMessage
from pprint import pprint

question = HumanMessage(content="Find fitness coaches in New York with 5k to 50k followers and a public email. Just list 2 coaches")

config = {"configurable": {"thread_id": "4"}}

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


{'messages': [HumanMessage(content='Find fitness coaches in New York with 5k to 50k followers and a public email. Just list 2 coaches', additional_kwargs={}, response_metadata={}, id='b84ac11e-b06b-439e-a40d-6841b2a45b4a'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'estimate_credits', 'arguments': '{"goal": 2}'}, '__gemini_function_call_thought_signatures__': {'call_1344569': 'EpsPCpgPAWkUfRMHw4b4VskRhfRAMiR1dQ6xIysibC3t3n3Zt5EuHrMMT043+0mM5csKTYNZrAU1aROf4ktEPb1F5nJ7LWFO5A4ydRg9CM7LtL3YSyhDe/TyCRbNVYcyhzLKHPYhqUqm09MGOm8T7R7Df9frZmJ16UPigcDPXBo7fhG3aDkokzVe5E/riyaYaGAFQ8qq5SGhjSzlsAQgnCAfb7UVGFBaULygN8jhf6UKp8pKVvB24lKBSb2VBlfBKr9n7huOFC7Vx2drYa7Aen2gKbz4sCKvhOUAHz/3GwTSDPndWbntzyMw65gDs92bZ6yEl1o/ove96xC6cQKfRCE0WVR4Zvq7OQ1FdvVw5KRwdMQh7UmSnKyBhDHCEkSVF5gc2oBs8J3pz8prBWk1ZR3rWLdbzY5k/fxZ4CPS4ffs732YfnZ199/wSQq6q0IVH1KijUkfAmP3rJ4E9MT0w9pI55YusRh6Paov9DgrsU8s4xQ9H15+phIF0EHUNvywGUj6LHFaDUy1X/1VPBW2/SdvL7dQwFDGh3fpXv2uqgH5krB6pSPzSL/BReO7io7XKtSLZ

In [13]:
question = HumanMessage(content="Look for any 1 Yoga instructor with minimum 50k followers in New York, who has posted atleast 1 post in last 7 days. Proceed with the search, no need for asking.")

response = await agent.ainvoke(
    {"messages": [question]},
    config=config
)

pprint(response)

Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


{'messages': [HumanMessage(content='Look for any 1 Yoga instructor with minimum 50k followers in New York, who has posted atleast 1 post in last 7 days.', additional_kwargs={}, response_metadata={}, id='cdef7823-435c-4446-9f2f-314ca215c44e'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'estimate_credits', 'arguments': '{"goal": 1}'}, '__gemini_function_call_thought_signatures__': {'call_544991': 'EoUMCoIMAWkUfRP6BzaH69iXOnFpEXsywalQU/2PY9sAP6OnkQottL1OyVjh1598mUbSm5PkDPBEsYvhPHBUiBFf00CaCCrNceGdUw4E+6FSMr+l7m0mn4QkT2Kz0/H/DY5iaBfJ2UP5Xf9CBZ39Ai1PQgf1MENMm8YmZc3uRF7PTwi98yJX1NLuYBiscDPa9Am025Oql/AImgJlSrJjdzop/FWCC8IV+tVdm8idPZIyWinEpiAFxHHXyOtyyXnwWKqgPuD5b74401cwPvlDqOjbu2X/+A/oq1/5apKzIZbODcFKlzjG7aEVofPlHeMO14yeiNL8yZgufEQV/Imcn2Whg85rAppOp/Ac5cTqYAS4lxnRlEA3rFomzL56q6ywllST/3GkcqL+gzAeqkNJ92r5YlpDHfGagz1aNMPfyDuwFAOGkpMP6kwtY2iokru7H0+zm0WH+aFn3vQgaG4MO5uKVjib84K3c4flRh6T7AnScs8qywFM8aaxLDD8gOQ+AvxRYHu/WwthsxCwau9bldMHPMSfofSXDkOgnU+zFbEsKy6RnZ6